# 02_CollectDAtasets_and_CloneAnnotation
Finalized and run on june 2nd 2026

In [ ]:
import anndata as ad
import scanpy as sc
import scanpy.external as sce
import scirpy as ir
import pandas as pd
import numpy as np
import seaborn as sb
import os
import muon as mu
import tempfile
from muon import prot as pt
from scipy.stats import binomtest
from scipy.stats import mannwhitneyu

# Helps to define plotting area
import matplotlib.pyplot as plt
from matplotlib import rcParams 
from matplotlib.pyplot import rc_context
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors

# set seed
np.random.seed(44)

#display options
pd.options.display.max_rows = 999
pd.options.display.max_columns = 999

# Helper functions from Mimi

In [ ]:
tcr_gene_prefixs = ['TRAV', 'TRAJ', 'TRAC', 'TRB', 'TRDV', 'TRDC', 'TRG']

def filter_tcr_genes(adata):
    non_tcr_genes = adata.var_names
    for prefix in tcr_gene_prefixs:
        non_tcr_genes = [el for el in non_tcr_genes if not el.startswith(prefix)]
    adata = adata[:, non_tcr_genes]
    return adata

def filter_high_var(adata, n):
    #sc.pp.highly_variable_genes(
    #adata,
    #flavor="seurat_v3",
    #n_top_genes=n,
    #layer="counts",
    #batch_key="pool",
    #subset=True,
    #span = 0.75) #default = 0.3. It's the fraction of cells used for estimation of hvg - if I have 150 cells in one sample but 150.000 cells in total then this sample becomes an issue. 
    
    sc.pp.highly_variable_genes(adata, n_top_genes=n)
    adata = adata[:, adata.var['highly_variable']].copy()
    return adata


def calculate_umap(adata, n_high_var=5000, remove_tcr_genes=True):
    """
    Calculates UMAP coordinates
    :param adata: AnnData object
    :param n_high_var: how many variable genes to use, None uses raw data
    :param remove_tcr_genes: whether to remove TCR genes
    :return: results stored in adata object
    """
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)            
    sc.tl.umap(adata_tmp)
    adata.obsm['X_umap'] = adata_tmp.obsm['X_umap']


def calculate_leiden(adata, resolution, n_high_var=5000, remove_tcr_genes=True):
    adata_tmp = adata.copy()

    if remove_tcr_genes:
        adata_tmp = filter_tcr_genes(adata_tmp)
    if n_high_var is not None:
        adata_tmp = filter_high_var(adata_tmp, n_high_var)

    sc.pp.neighbors(adata_tmp, n_pcs=20)
    sc.tl.leiden(adata_tmp, resolution=resolution)

    leiden_column_name = f"leiden_{resolution}"
    adata.obs[leiden_column_name] = adata_tmp.obs['leiden']

# Loading data
See the following for a discussion on data concatenation in muon: https://github.com/scverse/muon/issues/127

In [ ]:
os.getcwd()

In [ ]:
path_base = r"..\..\04_data_objects\01_preprocessed_scseq_scanpy\2025_12 - with EBV"

In [ ]:
adatas = []

# CSF data
# This will load NK_1/Pt.1/IIH, NK_2/Pt.4/MS + Pt.5/MS or HC, and NK_3/Pt.8/IIH
for i in range(1, 4):
    adata_tmp = mu.read(f'{path_base}\\NK_{i}.h5mu')
    adatas.append(adata_tmp)

# This will load NK_5/Pt.11/MS, NK_6/Pt.12/MS, NK_7/Pt.14
for i in range(5, 8):
    adata_tmp = mu.read(f'{path_base}\\NK_{i}.h5mu')
    adatas.append(adata_tmp)

# This will load NK_9 and NK_10
for i in ["9a", "9b", "10"]:
    adata_tmp = mu.read(f'{path_base}\\NK_{i}.h5mu')
    adatas.append(adata_tmp)

# PBMC data
# This will load NK_4
for i in ["x", "y", "z"]:
    adata_tmp = mu.read(f'{path_base}\\NK_4{i}.h5mu')
    adatas.append(adata_tmp)

# This will load NK_8
for i in ["x", "y", "z"]:
    adata_tmp = mu.read(f'{path_base}\\NK_8{i}.h5mu')
    adatas.append(adata_tmp)

# This will load NK_12
for i in ["1", "2", "3", "4"]:
    adata_tmp = mu.read(f'{path_base}\\NK_12_{i}.h5mu')
    adatas.append(adata_tmp)

# NK_11a (multiplexed) and NK_11b (
# This will load NK_11a (multiplex Pt. 20/MS and pt.21/HC) and NK_11b (pt.20/MS)
for i in ["a", "b"]:
    adata_tmp = mu.read(f'{path_base}\\NK_11{i}.h5mu')
    adatas.append(adata_tmp)

# Multimer data from NK 13
for i in range(1, 3):
    adata_tmp = mu.read(f'{path_base}\\NK13_{i}.h5mu')
    adatas.append(adata_tmp)

# Sample annotations, here called "pool"

In [ ]:
# Give annotations to samples where no CITEseq was included
adatas[0]["gex"].obs["pool"]="Pt1"
adatas[2]["gex"].obs["pool"]="Pt8"
adatas[3]["gex"].obs["pool"]="Pt11"
adatas[4]["gex"].obs["pool"]="Pt12"
adatas[5]["gex"].obs["pool"]="Pt14"
adatas[6]["gex"].obs["pool"]="Pt17"
adatas[7]["gex"].obs["pool"]="Pt18"
adatas[8]["gex"].obs["pool"]="Pt19"
adatas[19]["gex"].obs["pool"]="Pt20+21"
adatas[20]["gex"].obs["pool"]="Pt20"

## Multimer data already has a donor column

In [ ]:
# Give CSF samples compatible cohort groups
adatas[0]["gex"].obs["CohortGroup"]="Control"  #Pt1 NK1
adatas[1]["gex"].obs["CohortGroup"]="MS"       #Pt4 + 5 NK2
adatas[2]["gex"].obs["CohortGroup"]="Control"  #Pt8 NK3
adatas[3]["gex"].obs["CohortGroup"]="MS"       #Pt11 NK5
adatas[4]["gex"].obs["CohortGroup"]="MS"       #Pt12 NK6
adatas[5]["gex"].obs["CohortGroup"]="MS"       #Pt14 NK7
adatas[6]["gex"].obs["CohortGroup"]="MS"       #Pt17 NK9a
adatas[7]["gex"].obs["CohortGroup"]="Control"  #Pt18 NK9b
adatas[8]["gex"].obs["CohortGroup"]="Control"  #Pt19 NK10
adatas[19]["gex"].obs["CohortGroup"]="Multiplexed" #Pt20+21
adatas[20]["gex"].obs["CohortGroup"]="MS"      #Pt20
adatas[21]["gex"].obs["CohortGroup"]="Multimer"
adatas[22]["gex"].obs["CohortGroup"]="Multimer"

Data cleaning - streamline the var.names for the protein data modality

# Collect datasets
This part uses an updated concatenate function from AnnData (ad.concat) instead of the outdata version ad.concatenate(). 
Here, a dictionary of AnnData objects is defined for each modality, and indexes are made unique before concatenation by the "label" and "index_unique" variables. 
The index_unique variable works by taking the specified "batch" name from the dictionary input, and adding it to the cell-index name like the example shown below in comments. 

Join outer is chosen to avoid loss of genes. NK_2 is a very small dataset compared to most other datasets, join="inner" would remove a lot of useful genes from the merged datasets, because these genes just weren't present in NK_2. 

axis="0" refers to concatenation of "cell-level information" across the object i.e. additional cells are added to the final AnnData object. 

Source: https://anndata.readthedocs.io/en/latest/generated/anndata.concat.html

In [ ]:
number_of_elements = len(adatas)
print("Number of elements in the list:", number_of_elements)
# adatas

In [ ]:
mdata = []
airmdata = []
protdata = []

#concatenate GEX - outer-join will fill the gene expression matrix with 0s - I think this is an okay assumption
mdata = ad.concat({"NK_1":  adatas[0]["gex"], 
                   "NK_2":  adatas[1]["gex"], 
                   "NK_3":  adatas[2]["gex"], 
                   "NK_5":  adatas[3]["gex"],
                   "NK_6":  adatas[4]["gex"],
                   "NK_7":  adatas[5]["gex"],
                  "NK_9a":  adatas[6]["gex"],
                  "NK_9b":  adatas[7]["gex"],
                  "NK_10":  adatas[8]["gex"],
                   "NK_4x": adatas[9]["gex"], 
                   "NK_4y": adatas[10]["gex"], 
                   "NK_4z": adatas[11]["gex"], 
                   "NK_8x": adatas[12]["gex"], 
                   "NK_8y": adatas[13]["gex"], 
                   "NK_8z": adatas[14]["gex"],
                   "NK_12_1": adatas[15]["gex"], 
                   "NK_12_2": adatas[16]["gex"], 
                   "NK_12_3": adatas[17]["gex"],
                   "NK_12_4": adatas[18]["gex"],
                   "NK_11a": adatas[19]["gex"], 
                   "NK_11b": adatas[20]["gex"], 
                   "NK13_1":  adatas[21]["gex"], 
                   "NK13_2":  adatas[22]["gex"]}, 
                  join="outer", index_unique="-", axis=0, label="batch")

#concatenate AIRR - join inner on axis 0 to add adata with the same headers to each other.  
airmdata = ad.concat({"NK_1":  adatas[0]["airr"], 
                   "NK_2":  adatas[1]["airr"], 
                   "NK_3":  adatas[2]["airr"], 
                   "NK_5":  adatas[3]["airr"],
                   "NK_6":  adatas[4]["airr"],
                   "NK_7":  adatas[5]["airr"],
                   "NK_9a":  adatas[6]["airr"],
                   "NK_9b":  adatas[7]["airr"],
                   "NK_10":  adatas[8]["airr"],
                   "NK_4x": adatas[9]["airr"], 
                   "NK_4y": adatas[10]["airr"], 
                   "NK_4z": adatas[11]["airr"], 
                   "NK_8x": adatas[12]["airr"], 
                   "NK_8y": adatas[13]["airr"], 
                   "NK_8z": adatas[14]["airr"],
                   "NK_12_1": adatas[15]["airr"], 
                   "NK_12_2": adatas[16]["airr"], 
                   "NK_12_3": adatas[17]["airr"],
                   "NK_12_4": adatas[18]["airr"],
                   "NK_11a": adatas[19]["airr"], 
                   "NK_11b": adatas[20]["airr"],
                    "NK13_1":  adatas[21]["airr"], 
                   "NK13_2":  adatas[22]["airr"]}, 
                  join="inner", index_unique="-", axis=0, label="batch")

# Protein data - saved as vars . and outer join on obs axis will cause common columns to be kept, and NA values introduced elsewhere. 
protdata = ad.concat({"NK_2": adatas[1]["prot"],
                  "NK_4x": adatas[9]["prot"], 
                   "NK_4y": adatas[10]["prot"], 
                   "NK_4z": adatas[11]["prot"], 
                   "NK_8x": adatas[12]["prot"], 
                   "NK_8y": adatas[13]["prot"], 
                   "NK_8z": adatas[14]["prot"],
                   "NK_12_1": adatas[15]["prot"], 
                   "NK_12_2": adatas[16]["prot"], 
                   "NK_12_3": adatas[17]["prot"],
                   "NK_12_4": adatas[18]["prot"], 
                    "NK13_1": adatas[21]["prot"],
                  "NK13_2": adatas[22]["prot"]},
                     join="outer", index_unique="-", axis=0, label="batch")

## Collect all data into 1 object

In [ ]:
#Combine all
mdata = mu.MuData({'gex': mdata, 'airr': airmdata, "prot": protdata})

## Clean-up and align the 3 modalities
- Make unique indexes across the entire project
- Make sure that airr and prot knows which cells has gex.

In [ ]:
mdata

In [ ]:
mdata["gex"].obs["batch"].value_counts()

In [ ]:
###Clean-up.
#Make sure only the indexes recorded in GEX are used in the other modalities too
index_list = mdata["gex"].obs.index.tolist()

In [ ]:
## define a cell index
mdata["airr"].obs["cell_index"] = mdata["airr"].obs.index
mdata["prot"].obs["cell_index"] = mdata["prot"].obs.index

## ".isin" using indexes does not seem to work on a global level, but if I annotate a column and then filter in place, maybe then it works. 
mdata["airr"].obs["filter_index"] = pd.Categorical(np.where(mdata["airr"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))
mdata["prot"].obs["filter_index"] = pd.Categorical(np.where(mdata["prot"].obs["cell_index"].isin(index_list), "GEX_present", "GEX_absent"))

In [ ]:
#Enforce that mdata["airr"].obs and mdata["prot"].obs only contain information on those indexes that exists in GEX
mu.pp.filter_obs(mdata["airr"], 'filter_index', lambda x: x == "GEX_present")
mu.pp.filter_obs(mdata["prot"], 'filter_index', lambda x: x == "GEX_present")

In [ ]:
mdata.update()

# Sample annotation continued

In [ ]:
## Correct pool - add a suffix to each NK_12 
mdata["gex"].obs["pool"].drop_duplicates()

In [ ]:
## Find the batch column categories & indexes that contain "NK_12". Add _PBMC to pool and add the correct "t". This error stems from naming things slightly different in CellRanger for this run
mask = mdata["gex"].obs["batch"].str.contains("NK_12", na=False)
mdata["gex"].obs.loc[mask, "pool"] = (
    mdata["gex"].obs.loc[mask, "pool"]
    .astype(str)
    .str.slice(0, 1) + "t" + mdata["gex"].obs.loc[mask, "pool"].astype(str).str.slice(1) + "_PBMC"
)

In [ ]:
## Check corrected pool
mdata["gex"].obs["pool"].drop_duplicates()

In [ ]:
## Correct pool for NK13_1 + 2. The donor information is present in the "donor" column
mdata["gex"].obs["pool"] = pd.Categorical(np.where(mdata["gex"].obs["pool"].str.contains("HTO"), mdata["gex"].obs["donor"], mdata["gex"].obs["pool"])) 

In [ ]:
## Check corrected pool
mdata["gex"].obs["pool"].drop_duplicates()

In [ ]:
mdata.update()

# Quick visuals - cells per sample

In [ ]:
samples = list(pd.DataFrame(mdata["gex"].obs['pool'].value_counts()).reset_index()['pool'])
counts = list(pd.DataFrame(mdata["gex"].obs['pool'].value_counts()).reset_index()['count'])

plt.figure(figsize=(10,6))
plt.bar(samples, counts, label=samples)
plt.ylabel('cells')
plt.xlabel('sample')
plt.title('Cells per Sample')
plt.tick_params(axis='x', labelrotation=90)
plt.show()

We also perform simple filtering to make sure we have no features with zero counts (this is necessary because by selecting a subset of samples we may have removed all the cells which expressed a particular feature).

In [ ]:
mdata.update()

In [ ]:
sc.pl.scatter(mdata['prot'], x="CD4_adt", y="CD8_adt", color="cd4cd8_cs_assignment")

# Scsplit annotations

In [ ]:
# Read ithe probability of each cell belonging to each sample
split_nk11 = pd.read_csv(r'..\..\03_annotation\05_Scsplit_outs\NK11a\scSplit_P_s_c.csv')

# Determine color and label
def get_color_and_label(row):
    values = row[['0', '1', '2']]
    if (values > 0.75).any():
        max_col = values.idxmax()
        color = {'0': 'red', '1': 'blue', '2': 'green'}[max_col]
        return pd.Series([color, max_col])
    return pd.Series(['gray', 'Low'])

split_nk11[['color', 'label']] = split_nk11.apply(get_color_and_label, axis=1)

# Count assigned labels (exclude "Low")
label_counts = split_nk11[split_nk11['label'] != 'Low']['label'].value_counts().sort_index()

# Create side-by-side plots
fig, axes = plt.subplots(1, 3, figsize=(12, 4), gridspec_kw={'width_ratios': [3, 3, 2]})

# --- Left: Scatter plot ---
axes[0].scatter(split_nk11['0'], split_nk11['2'], c=split_nk11['color'], edgecolor='black', s=100)

# Custom legend
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', label='SNG-0 label', markerfacecolor='red', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='DBL-1 label', markerfacecolor='blue', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='SNG-2 label', markerfacecolor='green', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Low Probability', markerfacecolor='gray', markersize=10)
]
axes[0].legend(handles=legend_handles, title="Color Legend", loc='upper right')
axes[0].set_xlabel('SNG-0')
axes[0].set_ylabel('SNG-2')
axes[0].set_title('XY Plot with Conditional Coloring')
axes[0].grid(True)

# --- Middle: Scatter plot ---
axes[1].scatter(split_nk11['0'], split_nk11['1'], c=split_nk11['color'], edgecolor='black', s=100)

# Custom legend
legend_handles = [
    plt.Line2D([0], [0], marker='o', color='w', label='SNG-0 label', markerfacecolor='red', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='DBL-1 label', markerfacecolor='blue', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='SNG-2 label', markerfacecolor='green', markersize=10),
    plt.Line2D([0], [0], marker='o', color='w', label='Low Probability', markerfacecolor='gray', markersize=10)
]
axes[1].legend(handles=legend_handles, title="Color Legend", loc='upper right')
axes[1].set_xlabel('SNG-0')
axes[1].set_ylabel('DBL-1')
axes[1].set_title('XY Plot with Conditional Coloring')
axes[1].grid(True)


# --- Right: Bar plot ---
axes[2].bar(label_counts.index, label_counts.values, color=['red', 'blue', 'green'])
axes[2].set_xticks([0, 1, 2])
axes[2].set_xticklabels(['SNG-0', 'DBL-1', 'SNG-2'])
axes[2].set_ylabel('Number of Cells')
axes[2].set_title('Cell Count per Label')
axes[2].grid(axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Read the "scSplit_result.csv"
split_labels_nk11 = pd.read_csv(r'..\..\03_annotation\05_Scsplit_outs\NK11a\scSplit_result.csv',
                        sep='\t')

# Count assigned labels (exclude "Low")
label_counts = split_labels_nk11['Cluster'].value_counts().sort_index()
label_counts

In [ ]:
#What does the result dataframe look like? 
print(split_labels_nk11.head(2))

# Modify the index of split_nk11 to match the indexes in the sc-seq object
split_labels_nk11["index_placeholder"] = split_labels_nk11["Barcode"] + "-NK_11a"  
split_labels_nk11.set_index("index_placeholder", inplace=True)
scsplit_tomerge = split_labels_nk11[["Cluster"]]
scsplit_tomerge = scsplit_tomerge.rename(columns={"Cluster": "scSplit_cluster"})
print(scsplit_tomerge.head(2))

#join (left join by default)
mdata["gex"].obs = mdata["gex"].obs.join(scsplit_tomerge) #Left_join unto mdata
mdata["gex"].obs['scSplit_cluster'] = mdata["gex"].obs['scSplit_cluster'].fillna("Irrellevant")
print(mdata["gex"].obs.head(2))

In [ ]:
# Define donor labels
## Create a new column with the first 4 characters of 'Column1'
# airr_export["donor"] = airr_export["pool"].str[:4]
mdata["gex"].obs["donor"] = mdata["gex"].obs["pool"].str.replace("_PBMC", "", regex=False)

## Remove all occurrences of "_" from the strings in the column
mdata["gex"].obs["donor"] = mdata["gex"].obs["donor"].str.replace("_", "", regex=False)

mdata["gex"].obs["donor"].unique()

In [ ]:
#Confusion matrix - clone_handle vs scSplit
confusion_matrix = pd.crosstab(
        mdata["gex"].obs["donor"],
    mdata["gex"].obs["scSplit_cluster"],
    rownames=['donor'],
    colnames=['scSplit_cluster']
)

confusion_matrix

In [ ]:
mdata["gex"].obs["batch"].value_counts()

## Demultiplex sample annotations using Scplit

In [ ]:
### Cell counts per pool
group_counts = mdata["gex"].obs["pool"].value_counts().sort_values(ascending=False)

# Plot
plt.figure(figsize=(8, 5))
plt.bar(group_counts.index, group_counts.values, color='gray')  # uniform color
plt.xlabel("Pool")
plt.ylabel("Number of Cells")
plt.title("Cell Counts per Pool (Descending Order)\nNext: Reassign Pt20+Pt21")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
## Save the old pool first
mdata["gex"].obs["pool_ori"] = mdata["gex"].obs["pool"].copy()

In [ ]:
#Redefine "pool" according to the information found by scsplit
mdata["gex"].obs["pool"] = pd.Categorical(
    np.where(mdata["gex"].obs["scSplit_cluster"] == "SNG-0", "Pt20",
        np.where(mdata["gex"].obs["scSplit_cluster"] == "SNG-2", "Pt21",
                 np.where(mdata["gex"].obs["scSplit_cluster"] == "DBL-1", "Pt20+Pt21", 
                          np.where((mdata["gex"].obs["donor"] == "Pt20+21") & (mdata["gex"].obs["scSplit_cluster"] == "Irrellevant"), "Unassigned", mdata["gex"].obs["pool"])))))

mdata.update()

In [ ]:
### Cell counts per pool
group_counts = mdata["gex"].obs["pool"].value_counts().sort_values(ascending=False)

# Plot
plt.figure(figsize=(8, 5))
plt.bar(group_counts.index, group_counts.values, color='gray')  # uniform color
plt.xlabel("Pool")
plt.ylabel("Number of Cells")
plt.title("Cell Counts per Pool (Descending Order)\nAfter demultiplexing Pt20 and Pt21")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Export NK11a cells for doublet supporting analysis

In [ ]:
mdata["gex"].obs[mdata["gex"].obs["batch"] == "NK_11a"][["pool", "batch"]].value_counts()

In [ ]:
# mu.pp.filter_obs(mdata["gex"], 'batch', lambda x: (x == "NK_11a"))
# mdata["gex"].obs["batch"] = mdata["gex"].obs["batch"].cat.remove_unused_categories()
# mdata["gex"].obs["pool"] = mdata["gex"].obs["pool"].cat.remove_unused_categories()
# mdata.update()

# mdata.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_MixedCSF_NK_11a.h5mu")

# Remove technical doublets from Scplit

In [ ]:
# mu.pp.filter_obs(mdata["gex"], 'scSplit_cluster', lambda x: (x != "DBL-1"))
mu.pp.filter_obs(mdata["gex"], 'pool', lambda x: (x != "Pt20+Pt21"))
mu.pp.filter_obs(mdata["gex"], 'pool', lambda x: (x != "Unassigned"))
mdata["gex"].obs["pool"] = mdata["gex"].obs["pool"].cat.remove_unused_categories()
# mdata["gex"].obs["donor"] = mdata["gex"].obs["donor"].cat.remove_unused_categories()
mdata.update()

In [ ]:
mdata["gex"].obs["batch"].value_counts()

In [ ]:
### Cell counts per pool
group_counts = mdata["gex"].obs["pool"].value_counts().sort_values(ascending=False)

# Plot
plt.figure(figsize=(8, 5))
plt.bar(group_counts.index, group_counts.values, color='gray')  # uniform color
plt.xlabel("Pool")
plt.ylabel("Number of Cells")
plt.title("Cell Counts per Pool (Descending Order)\nAfter demultiplexing Pt20 and Pt21")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Define the donor column

In [ ]:
## DEFINE DONOR IN "gex" first. 
mdata["gex"].obs["donor"] = mdata["gex"].obs["pool"].str.replace("_PBMC", "", regex=False)
mdata["gex"].obs["donor"] = mdata["gex"].obs["donor"].str.replace("_", "", regex=False)

## TRANSFER TO AIR
mdata["airr"].obs["donor"] = mdata["gex"].obs["donor"]
mdata.update()

# # Generate a donor-wise column
# mdata["airr"].obs["pool"] = mdata["gex"].obs["pool"]
# mdata["airr"].obs["donor"] = mdata["airr"].obs["pool"].str.replace("_PBMC", "", regex=False)

# ## Remove all occurrences of "_" from the strings in the column
# mdata["airr"].obs["donor"] = mdata["airr"].obs["donor"].str.replace("_", "", regex=False)

mdata["airr"].obs["donor"].drop_duplicates().tolist()

In [ ]:
mdata["gex"].obs["donor"].drop_duplicates().tolist()

In [ ]:
mdata["gex"].obs["pool"].drop_duplicates().tolist()

# Perform repetoire analysis -- All T cells cells
Clone definition and annotation to vdjdb

In [ ]:
## Define clonotypes according to the funtion Mimi uses
## Takes a while
#TCR steps %%%%%%%%%%%%%%%%%%%%%%%%%%%
ir.pp.index_chains(mdata)
ir.tl.chain_qc(mdata)    #Chain QC

### Clone definition

In [ ]:
# We must define a grouping variable for clone
# Define the new column based on the value observed in chain_pairing
## This caused clones seen in 1 multiplet to be completely unlinked from their singlet counter parts (probably a good thing for clone_size estimation)
mdata["airr"].obs["clonotype_groups"] = mdata["airr"].obs["chain_pairing"].apply(lambda x: 'two full chains' if x == 'two full chains' else 'single chains')


In [ ]:
ir.pp.ir_dist(mdata,  metric='identity', sequence='aa')   #First you have to calculate sequence distances

#This will add a clone_id column to adata.obs.
ir.tl.define_clonotype_clusters(mdata, 
                                sequence='aa',            #Each clonotype is defined by it's amino acid (aa) identity
                                metric='identity',        #Identity refers to exact matches between TCRs from different cells. OPTIONAL: TCRdist related matching. 
                                receptor_arms='all',      #TRA and TRB will both need to be considered for a clone_id
                                within_group=["clonotype_groups"],   #SEE COMMENTS BELOW. 
                                dual_ir='any',            #The order in which TRA_1, TRA_2, TRB_1, and TRB_2 doesn't matter. #Any dual receptor information will be used to group cells into clones
                                key_added='clone_id')  #A column name. 

# "within_group = ["clonotype_groups"]" will make it so that clonotype_assignment is done for each group separately (two-chain TCRs vs the rest)
# This removes the "bridging effect" that I've observed, whereby clone ids grow exponentially as more CDR3s are associated with the same clone. 



#### Remove observations we did not find a donor for from the AIRR modality

In [ ]:
mdata["airr"].obs["donor"] = mdata["airr"].obs["donor"].fillna("null")

In [ ]:
mu.pp.filter_obs(mdata["airr"], 'donor', lambda x: (x != "null"))
mdata.update()

# Match to a database based on identity

scirpy function  
scirpy.pp.ir_dist() Compute sequence-based distance matrices within your dataset

scirpy.tl.ir_query() For each cell, identify matching entries in a reference database.

scirpy.tl.ir_query_annotate()  Transfer annotations from reference database to adata.obs.

scirpy.tl.ir_query_annotate_df()  Return a dataframe with all matching annotations.

In [ ]:
#Epitope annotation
#Takes a long time
vdjdb = ir.datasets.vdjdb()
ir.pp.ir_dist(mdata, reference=vdjdb, metric="identity", sequence="aa")  #We can look at distances between a dataset and a reference (here VDJdb)
ir.tl.ir_query(
    mdata,
    reference=vdjdb,      #The reference we wish to match to
    metric="identity",    #uses identity to find matches.
    sequence="aa",        #uses amino acid sequences
    receptor_arms="all",    #matches between both TRA and TRB is required
    dual_ir="primary_only",  #annotation is not split into whether the epitope is specific to primary or secondary. Best to focus on the primary then.
)

In [ ]:
ir.tl.ir_query_annotate(
    mdata,
    vdjdb,
    metric="identity",
    sequence="aa",
    include_ref_cols=["antigen.species", "antigen.gene" ,"antigen.epitope", "mhc.a", "mhc.b"],
    strategy="most-frequent",
)

In [ ]:
mdata["airr"].obs["antigen.species"].value_counts()

## Airr information transfer

In [ ]:
#Because the default behavior is not to save "aa" information in the awkward array, we manually add this information afterwards.
mdata["airr"].obs["IR_VJ_1_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VJ_1"])
mdata["airr"].obs["IR_VJ_2_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VJ_2"])
mdata["airr"].obs["IR_VDJ_1_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VDJ_1"])
mdata["airr"].obs["IR_VDJ_2_junction_aa"] = ir.get.airr(mdata, "junction_aa", ["VDJ_2"])

mdata["airr"].obs["IR_VJ_1_junction_nt"] = ir.get.airr(mdata, "junction", ["VJ_1"])
mdata["airr"].obs["IR_VJ_2_junction_nt"] = ir.get.airr(mdata, "junction", ["VJ_2"])
mdata["airr"].obs["IR_VDJ_1_junction_nt"] = ir.get.airr(mdata, "junction", ["VDJ_1"])
mdata["airr"].obs["IR_VDJ_2_junction_nt"] = ir.get.airr(mdata, "junction", ["VDJ_2"])


mdata["airr"].obs["IR_VJ_1_v_call"] = ir.get.airr(mdata, "v_call", ["VJ_1"])
mdata["airr"].obs["IR_VJ_2_v_call"] = ir.get.airr(mdata, "v_call", ["VJ_2"])
mdata["airr"].obs["IR_VDJ_1_v_call"] = ir.get.airr(mdata, "v_call", ["VDJ_1"])
mdata["airr"].obs["IR_VDJ_2_v_call"] = ir.get.airr(mdata, "v_call", ["VDJ_2"])

mdata["airr"].obs["IR_VJ_1_j_call"] = ir.get.airr(mdata, "j_call", ["VJ_1"])
mdata["airr"].obs["IR_VJ_2_j_call"] = ir.get.airr(mdata, "j_call", ["VJ_2"])
mdata["airr"].obs["IR_VDJ_1_j_call"] = ir.get.airr(mdata, "j_call", ["VDJ_1"])
mdata["airr"].obs["IR_VDJ_2_j_call"] = ir.get.airr(mdata, "j_call", ["VDJ_2"])

mdata["airr"].obs["IR_VDJ_1_d_call"] = ir.get.airr(mdata, "d_call", ["VDJ_1"])
mdata["airr"].obs["IR_VDJ_2_d_call"] = ir.get.airr(mdata, "d_call", ["VDJ_2"])

#We also need the pool information within this obs object
mdata["airr"].obs["pool"] = mdata["gex"].obs["pool"]

#Set clone_id to NaN if a chain is missing
mdata["airr"].obs.loc[mdata["airr"].obs['IR_VJ_1_junction_aa'].isna(), 'clone_id'] = np.nan
mdata["airr"].obs.loc[mdata["airr"].obs['IR_VDJ_1_junction_aa'].isna(), 'clone_id'] = np.nan

#Transfer labels
mdata["gex"].obs["clone_id"] = mdata["airr"].obs["clone_id"].copy()
mdata["prot"].obs["clone_id"] = mdata["airr"].obs["clone_id"].copy()

#Antigen information too
mdata["gex"].obs["antigen.species"] = mdata["airr"].obs["antigen.species"]
mdata["gex"].obs["antigen.gene"] = mdata["airr"].obs["antigen.gene"]
mdata["gex"].obs["antigen.epitope"] = mdata["airr"].obs["antigen.epitope"]

In [ ]:
mdata.update()

# Export clone assignments

In [ ]:
export_df = mdata["gex"].obs[["clone_id", "chain_pairing", "donor", "antigen.species", "antigen.gene", "antigen.epitope"]]
export_df = export_df.rename(columns={"donor": "donor_collect"})
export_df.to_csv("../../04_data_objects/06_preprocessed_v2/2026_6_Clone_assignment.csv")

In [ ]:
export_df.head(2)

In [ ]:
sum_df = mdata["gex"].obs[["batch", "clone_id"]]
sum_df["catbatch"] = pd.Categorical(
    np.where(sum_df["batch"].str.contains("NK13"), "Multimers", "Main dataset"))

# For each clone: how many distinct catbatch categories does it appear in?
clone_cat_counts = (
    sum_df
    .groupby("clone_id")["catbatch"]
    .nunique()
)

# Clones present in both categories
clones_in_both = clone_cat_counts[clone_cat_counts == 2]

n_clones_in_both = len(clones_in_both)
n_clones_in_both

# Export the Main dataset

In [ ]:
mdata["gex"].obs["batch"].value_counts()

In [ ]:
#Recalculate scrublet on a global level.
main_dataset = mdata.copy()
mu.pp.filter_obs(main_dataset["gex"], "batch", lambda x: (x != "NK13_1")) 
mu.pp.filter_obs(main_dataset["gex"], "batch", lambda x: (x != "NK13_2")) 
main_dataset.update()

main_dataset["gex"].obs["batch"] = main_dataset["gex"].obs["batch"].cat.remove_unused_categories()

In [ ]:
main_dataset.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_main_dataset_raw.h5mu")

In [ ]:
main_dataset["gex"].obs["batch"].value_counts()

# Export the multimer dataset

In [ ]:
#Recalculate scrublet on a global level.
multimer_dataset = mdata.copy()
mu.pp.filter_obs(multimer_dataset["gex"], "batch", lambda x: (x.isin(["NK13_1", "NK13_2"])))
multimer_dataset.update()

multimer_dataset["gex"].obs["batch"] = multimer_dataset["gex"].obs["batch"].cat.remove_unused_categories()

In [ ]:
multimer_dataset.write(filename="../../04_data_objects/06_preprocessed_v2/2026_6_multimer_dataset_raw.h5mu")

In [ ]:
multimer_dataset["gex"].obs["batch"].value_counts()